## Local Setup

Use the local project directory for data, utility code, and saved outputs, then train.
If a local ESM-2 snapshot exists under `hf_models/`, the notebook uses it; otherwise the helper falls back to the standard Hugging Face resolution path.


In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents, cwd / 'XAllergen2.0']
search_roots.extend(path for path in cwd.iterdir() if path.is_dir())

PROJECT_ROOT = None
for candidate in search_roots:
    if (candidate / 'data').exists() and (candidate / 'finetune_notebook_utils.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f'Could not locate project root from {cwd}. Expected a directory containing data/ and finetune_notebook_utils.py.'
    )

MODEL_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Local directories ready:')
print(f'  Project root: {PROJECT_ROOT}')
print(f'  Models: {MODEL_DIR}')
print(f'  Results: {RESULTS_DIR}')


In [ ]:
import os
from pathlib import Path

cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents, cwd / 'XAllergen2.0']
search_roots.extend(path for path in cwd.iterdir() if path.is_dir())

PROJECT_ROOT = None
for candidate in search_roots:
    if (candidate / 'data').exists() and (candidate / 'finetune_notebook_utils.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f'Could not locate project root from {cwd}. Expected a directory containing data/ and finetune_notebook_utils.py.'
    )

DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
LOCAL_ESM2_DIR = PROJECT_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'
for d in [MODEL_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

utils_path = PROJECT_ROOT / 'finetune_notebook_utils.py'
if not utils_path.exists():
    raise FileNotFoundError(f'Expected local utility script at {utils_path}')

required_data_files = [
    'deepalgpro_test_cleaned.csv',
    'deepalgpro_train_cleaned.csv',
    'negatives_splitA.csv',
    'negatives_splitB.csv',
    'positives_splitA.csv',
    'positives_splitB.csv',
]
missing_data_files = [fname for fname in required_data_files if not (DATA_DIR / fname).exists()]
if missing_data_files:
    raise FileNotFoundError('Missing local data files
' + '
'.join(f'  {fname}' for fname in missing_data_files))

if LOCAL_ESM2_DIR.exists():
    os.environ['XALLERGEN_HF_MODEL_DIR'] = str(LOCAL_ESM2_DIR)
    print(f'Using local ESM-2 snapshot: {LOCAL_ESM2_DIR}')
else:
    os.environ.pop('XALLERGEN_HF_MODEL_DIR', None)
    print('No local ESM-2 snapshot found under hf_models/; using default Hugging Face resolution.')

print('Local setup complete:')
print(f'  Project root: {PROJECT_ROOT}')
print(f'  Utility script: {utils_path}')
print(f'  Data dir: {DATA_DIR}')
print(f'  Model dir: {MODEL_DIR}')
print(f'  Results dir: {RESULTS_DIR}')


# Fine-tuned ESM-2 baseline for DeepAlgPro

This notebook trains a fine-tuned `esm2_t6_8M_UR50D` allergenicity classifier on `data/deepalgpro_train_cleaned.csv` and evaluates it on the held-out `data/deepalgpro_test_cleaned.csv`.

Architecture:
- Backbone: `esm2_t6_8M_UR50D`, unfrozen and fine-tuned throughout training.
- Attention pooling: `Linear(embed_dim -> 1)` per residue, softmax-normalized across sequence length, then a weighted sum of residue embeddings. The resulting per-residue weights are kept as an attribution signal.
- Classification head: `Linear(embed_dim -> 128) -> ReLU -> Dropout(0.3) -> Linear(128 -> 1)`.
- Training loss: `BCEWithLogitsLoss`; inference uses `sigmoid`.

Split strategy:
- `deepalgpro_train_cleaned.csv` is split into train and validation with a stratified random 90/10 split.
- `sklearn.model_selection.train_test_split` is used with `stratify=df["label"]` and `random_state=42`.
- `deepalgpro_test_cleaned.csv` remains fully held out until final evaluation.

In [ ]:
import json
import math
import sys
import time as _time
from pathlib import Path

# Add project root to path so finetune_notebook_utils is importable
cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents, cwd / 'XAllergen2.0']
search_roots.extend(path for path in cwd.iterdir() if path.is_dir())

PROJECT_ROOT = None
for candidate in search_roots:
    if (candidate / 'data').exists() and (candidate / 'finetune_notebook_utils.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f'Could not locate project root from {cwd}. Expected a directory containing data/ and finetune_notebook_utils.py.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from finetune_notebook_utils import (
    DROPOUT,
    ESM_MODEL_NAME,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    RANDOM_STATE,
    THRESHOLD,
    FinetunedESMAllergenClassifier,
    build_tokenizer,
    compute_attention_weights,
    compute_integrated_gradients,
    load_finetune_checkpoint,
    normalize_scores,
    seed_everything,
)

import matplotlib
matplotlib.use('Agg')  # headless-safe on local/Jupyter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# ── Hyperparameters ───────────────────────────────────────────────────────
BATCH_SIZE    = 24
EPOCHS        = 5
PATIENCE      = 5
LEARNING_RATE = 8e-6
WEIGHT_DECAY  = 1e-4

# ── Local paths ───────────────────────────────────────────────────────────
DATA_DIR    = PROJECT_ROOT / 'data'
MODEL_DIR   = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'

TRAIN_CSV       = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV        = DATA_DIR / 'deepalgpro_test_cleaned.csv'
CHECKPOINT_PATH = MODEL_DIR / 'finetuned_esm2_8e-6.pt'
METRICS_PATH    = RESULTS_DIR / 'finetuned_esm2_8e-6_metrics.json'

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [35]:
train_df = pd.read_csv(TRAIN_CSV).copy()
test_df = pd.read_csv(TEST_CSV).copy()
for df in [train_df, test_df]:
    df["sequence_id"] = df["sequence_id"].astype(str)
    df["sequence"] = df["sequence"].astype(str).str.strip().str.upper()
    df["label"] = df["label"].astype(int)

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df["label"],
)
train_split_df = train_split_df.reset_index(drop=True).copy()
val_split_df = val_split_df.reset_index(drop=True).copy()

print(f"Train sequences: {len(train_split_df):,}")
print(train_split_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())
print()
print(f"Validation sequences: {len(val_split_df):,}")
print(val_split_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())
print()
print(f"Held-out test sequences: {len(test_df):,}")
print(test_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())

Train sequences: 5,011
label
negative    2505
positive    2506

Validation sequences: 557
label
negative    279
positive    278

Held-out test sequences: 1,376
label
negative    688
positive    688


In [36]:
class SequenceDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        return {
            "sequence_id": row["sequence_id"],
            "sequence": row["sequence"],
            "label": int(row["label"]),
        }


tokenizer = build_tokenizer(HF_MODEL_NAME)


def collate_batch(batch: list[dict]) -> dict:
    sequences = [item["sequence"] for item in batch]
    encodings = tokenizer(
        sequences,
        add_special_tokens=False,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    return {
        "sequence_id": [item["sequence_id"] for item in batch],
        "sequence": sequences,
        "label": torch.tensor([item["label"] for item in batch], dtype=torch.float32),
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
    }


def move_batch_to_device(batch: dict, device: str) -> dict:
    moved = dict(batch)
    moved["input_ids"] = batch["input_ids"].to(device)
    moved["attention_mask"] = batch["attention_mask"].to(device)
    moved["label"] = batch["label"].to(device)
    return moved


train_loader = DataLoader(SequenceDataset(train_split_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_batch)
val_loader = DataLoader(SequenceDataset(val_split_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader = DataLoader(SequenceDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)

model = FinetunedESMAllergenClassifier(
    HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(DEVICE)
assert all(param.requires_grad for param in model.backbone.parameters())
trainable_params = [param for param in model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

print(f"Trainable parameter tensors: {len(trainable_params)}")
print(f"Backbone hidden size: {model.backbone.config.hidden_size}")


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameter tensors: 109
Backbone hidden size: 320


In [37]:
from typing import Optional, Tuple
from tqdm.auto import tqdm

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
    epoch: int,
) -> float:
    model.train()
    total_loss = 0.0
    total_examples = 0
    for batch in loader:  # ← plain loop, no inner tqdm
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        loss = criterion(outputs["logits"], batch["label"])
        loss.backward()
        optimizer.step()
        batch_size = batch["label"].shape[0]
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size
    return total_loss / max(total_examples, 1)


@torch.no_grad()
def predict(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    criterion: Optional[nn.Module] = None,
    desc: Optional[str] = None,
) -> Tuple[Optional[float], pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_examples = 0
    rows = []
    for batch in loader:  # ← plain loop, no inner tqdm
        batch = move_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        logits = outputs["logits"]
        probs = torch.sigmoid(logits)
        if criterion is not None:
            loss = criterion(logits, batch["label"])
            batch_size = batch["label"].shape[0]
            total_loss += float(loss.item()) * batch_size
            total_examples += batch_size
        for idx in range(len(batch["sequence_id"])):
            rows.append(
                {
                    "sequence_id": batch["sequence_id"][idx],
                    "sequence": batch["sequence"][idx],
                    "label": int(batch["label"][idx].item()),
                    "logit": float(logits[idx].item()),
                    "pred_prob": float(probs[idx].item()),
                    "pred_label": int(probs[idx].item() >= THRESHOLD),
                }
            )
    loss_value = None if criterion is None else total_loss / max(total_examples, 1)
    return loss_value, pd.DataFrame(rows)


def compute_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label"].to_numpy()
    y_prob = pred_df["pred_prob"].to_numpy()
    y_pred = pred_df["pred_label"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        "threshold": THRESHOLD,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "auroc": float(roc_auc_score(y_true, y_prob)),
        "auprc": float(average_precision_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else math.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }
    return metrics


history = []
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

_epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch", file=sys.stdout, dynamic_ncols=True)
for epoch in _epoch_bar:
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, epoch)
    val_loss, val_pred_df = predict(
        model, val_loader, DEVICE, criterion,
    )
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "esm_model_name": ESM_MODEL_NAME,
                "architecture_hyperparameters": {
                    "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT,
                },
                "training_history": history,
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    # ← print each epoch result as a plain line so you always see progress
    print(
        f"Epoch {epoch:>3}/{EPOCHS} | "
        f"train_loss={train_loss:.5f} | "
        f"val_loss={val_loss:.5f} | "
        f"best={best_epoch}"
    )
    _epoch_bar.set_postfix(
        train_loss=f"{train_loss:.5f}",
        val_loss=f"{val_loss:.5f}",
        best=best_epoch,
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print(f"Best validation loss: {best_val_loss:.5f} at epoch {best_epoch}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")

Training:   0%|          | 0/5 [00:00<?, ?epoch/s]

Epoch   1/5 | train_loss=0.69210 | val_loss=0.68739 | best=1
Epoch   2/5 | train_loss=0.68466 | val_loss=0.67901 | best=2
Epoch   3/5 | train_loss=0.67618 | val_loss=0.67051 | best=3
Epoch   4/5 | train_loss=0.66738 | val_loss=0.66163 | best=4
Epoch   5/5 | train_loss=0.65857 | val_loss=0.65236 | best=5
Best validation loss: 0.65236 at epoch 5
Checkpoint saved to: /content/XAllergen2.0/models/finetuned_esm2_8e-6.pt


In [38]:
model, checkpoint = load_finetune_checkpoint(
    CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
)

train_history_df = pd.DataFrame(checkpoint["training_history"])

val_loss, val_predictions_df = predict(model, val_loader, DEVICE, criterion)
test_loss, test_predictions_df = predict(model, test_loader, DEVICE, criterion)

test_metrics = compute_metrics(test_predictions_df)
test_metrics["test_loss"] = float(test_loss)
test_metrics["best_epoch"] = int(best_epoch)
test_metrics["n_test_sequences"] = int(len(test_predictions_df))

metrics_payload = {
    "esm_model_name": ESM_MODEL_NAME,
    "architecture_hyperparameters": {"hidden_dim": HIDDEN_DIM, "dropout": DROPOUT},
    "training": {
        "batch_size": BATCH_SIZE,
        "epochs_requested": EPOCHS,
        "early_stopping_patience": PATIENCE,
        "optimizer": "AdamW",
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
    },
    "validation_loss": float(val_loss),
    "test_metrics": test_metrics,
}

with METRICS_PATH.open("w") as handle:
    json.dump(metrics_payload, handle, indent=2)


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
print('Runtime outputs:')
print(f'  Checkpoint : {CHECKPOINT_PATH}')
print(f'  Metrics    : {METRICS_PATH}')
print(f'  (Will be copied to Drive in the final cell.)')


Runtime outputs:
  Checkpoint : /content/XAllergen2.0/models/finetuned_esm2_8e-6.pt
  Metrics    : /content/XAllergen2.0/results/finetuned_esm2_8e-6_metrics.json
  (Will be copied to Drive in the final cell.)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_history_df["epoch"], train_history_df["train_loss"], label="Train loss")
axes[0].plot(train_history_df["epoch"], train_history_df["val_loss"], label="Validation loss")
axes[0].set_title("Training and validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

cm = confusion_matrix(test_predictions_df["label"], test_predictions_df["pred_label"], labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_title("Held-out test confusion matrix")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
axes[1].set_xticklabels(["Negative", "Positive"])
axes[1].set_yticklabels(["Negative", "Positive"], rotation=0)

plt.tight_layout()
plt.show()

display(Markdown("## Test metrics"))
display(pd.DataFrame([{k: v for k, v in test_metrics.items() if k != 'confusion_matrix'}]).T.rename(columns={0: 'value'}))
print(f"Saved metrics JSON to: {METRICS_PATH}")

## Test metrics

,value
threshold,0.500000
accuracy,0.771076
auroc,0.895629
auprc,0.880288
f1,0.733728
precision,0.876768
recall,0.630814
specificity,0.911337
mcc,0.564831
test_loss,0.654721


Saved metrics JSON to: /content/XAllergen2.0/results/finetuned_esm2_8e-6_metrics.json


In [31]:
def select_attribution_examples(pred_df: pd.DataFrame) -> pd.DataFrame:
    selected_frames = []
    selected_ids = set()

    high_conf_tp = pred_df.loc[
        (pred_df["label"] == 1)
        & (pred_df["pred_label"] == 1)
        & (pred_df["pred_prob"] > 0.9)
    ]
    if len(high_conf_tp) > 0:
        tp_sample = high_conf_tp.sample(n=min(3, len(high_conf_tp)), random_state=RANDOM_STATE)
        selected_frames.append(tp_sample)
        selected_ids.update(tp_sample["sequence_id"])

    errors = pred_df.loc[pred_df["pred_label"] != pred_df["label"]]
    if len(errors) > 0:
        error_sample = errors.sample(n=min(2, len(errors)), random_state=RANDOM_STATE)
        selected_frames.append(error_sample)
        selected_ids.update(error_sample["sequence_id"])

    selected_df = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame(columns=pred_df.columns)

    if len(selected_df) < 5:
        remainder = pred_df.loc[~pred_df["sequence_id"].isin(selected_ids)]
        fill_n = min(5 - len(selected_df), len(remainder))
        if fill_n > 0:
            filler = remainder.sample(n=fill_n, random_state=RANDOM_STATE)
            selected_df = pd.concat([selected_df, filler], ignore_index=True)

    return selected_df.head(5).copy()


def compute_attention_and_ig(sequence: str) -> tuple[np.ndarray, np.ndarray]:
    attention_scores = normalize_scores(
        compute_attention_weights(model, tokenizer, sequence, DEVICE)
    )
    ig_scores = compute_integrated_gradients(
        model,
        tokenizer,
        sequence,
        DEVICE,
        steps=IG_STEPS,
        normalize=True,
    )
    return attention_scores, ig_scores


In [32]:
print('Saved locally:')
print(f'  {CHECKPOINT_PATH}')
print(f'  {METRICS_PATH}')


## Wrap-up

The fine-tuned ESM-2 checkpoint and held-out test metrics have been saved under the local project directory. Review the evaluation plots and test metrics above as the fine-tuning reference point.

The next step is dataset-scale attribution evaluation against IEDB epitope ground truth in `03_attribution_evaluation.ipynb`.
